# Artmatcher

This notebook demonstrates the artmatcher for determining whether a query image matches a reference artwork. The artmatcher makes use of a **Random Forest classifier**.

The classifier combines three feature categories:
- **Distribution features**: Statistics from embedding similarity scores (gaps, z-scores, decay rates)
- **ORB features**: Geometric verification using ORB keypoints and homography
- **Perceptual hashes**: Multiple hash distances (pHash, dHash, wHash, aHash)

These total 23 features (described below), which are fed into a **Random Forest** trained on labeled examples from the web UI.
The random forest outputs a confidence score, which is to say how confident the model is that two images are a 'match', given their features.

Main technologies used:
- [Scikit-learn Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees) - Ensemble classifier with probability calibration
  - a machine-learned decision tree for coming up with a confidence score based on a number of input metrics
- [OpenCV ORB](https://docs.opencv.org/4.x/d1/d89/tutorial_py_orb.html)
  - this detects 'features' (smalll regions of interest) in images, and stores them in a way that is invariant to scale and rotation
  - we search for matches: features that exist in both images, and feed information about these matches to the random forest classifier
- [Perceptual Hashing](https://github.com/JohannesBuchner/imagehash) - Image similarity using perceptual hashes
  - turns an image into a short code where similar-looking images get similar codes.
- (CLIP) Embedding's cosine similarities
  - _cosine similarity_ measures how much two vectors align

# Setup

### Dependencies

Import all necessary python libraries:

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from artmatcher.ml_classifier import FeatureExtractor, MLClassifier

### Configuration

Set the paths below to run on your own images and model.

- `IMAGES_DIR`: directory containing your images
- `QUERY_IMAGE` / `REFERENCE_IMAGE`: specific images to compare
- `TRAINED_MODEL_PATH`: path to a trained `.pkl` model — see [dataset.ipynb](dataset.ipynb) to create one

Run the code block below to load the paths to the set of example images and model:

_`IMAGES_DIR`_ should be an absolute path (starting with / on Unix or C:\\ on Windows) 

In [ ]:
# e.g. /Users/jane/Projects/glashelder/notebooks/artmatcher/examples/
IMAGES_DIR = Path("change me!")

The following constants are all defined based on the value you assigned to `IMAGES_DIR`. 

You don't need to change anything here.

In [ ]:
QUERY_IMAGE = IMAGES_DIR / "028pc5th0k_tiff.jpg"  # A glashelder photo
QUERY_IMAGE_2 = IMAGES_DIR / "1n7xm18d8t_tiff.jpg"

REFERENCE_IMAGE = (
    IMAGES_DIR / "joachim-patinir_rest-during-the-flight-to-egypt.jpg"
)  # A wikiart painting
REFERENCE_IMAGE_2 = (
    IMAGES_DIR
    / "notebooks/artmatcher/examples/ancient-greek-painting_boxer-head-by-myron--400.jpg"
)
TRAINED_MODEL_PATH = IMAGES_DIR / "trained-matcher-model.pkl"

---

# Part 1: Random Forest Classifier

The classifier decides whether a top embedding match is a **true match** or a false positive. It uses 23 engineered features across three categories:

```mermaid
flowchart TD
  subgraph Input
    emb[Embedding Similarities] --> dist[Distribution Features]
    imgs[Query + Reference Images] --> orb[ORB Features]
    imgs --> hash[Hash Features]
  end
  subgraph "Random Forest Classifier"
    dist --> feat[23-Feature Vector]
    orb --> feat
    hash --> feat
    feat --> rf[Calibrated Random Forest]
    rf --> decision["is_match + confidence"]
  end
```

**Workflow overview:**
1. Compute CLIP embedding similarities between query and reference database
2. Extract 23 features (distribution stats, ORB geometric verification, perceptual hashes)
3. Classify with a trained Random Forest → calibrated probability
4. Threshold the probability to decide match vs. no-match

## 1.1 The 23 Features

The classifier uses these features:

### Distribution (from similarities only):
- sim_0
- sim_1
- sim_2
- gap_01
- gap_02
- gap_12
- mean_1to9
- std_1to9
- sim_0_zscore
- decay_rate
- sim_min
- sim_range
- poly_diff
- sim_ratio_01

### ORB geometric (requires images):
- orb_num_matches
- orb_num_inliers
- orb_confidence
- orb_warp_stretch
- inlier_match_ratio

### Perceptual hashes (requires images):
- phash_distance
- dhash_distance
- whash_distance
- ahash_distance

## 1.2 Feature Extraction

Extract features from similarity scores and image pairs.

**In production**, similarities come from an embedding model (CLIP, DINOv2, or Qwen3VL):
1. Encode the query image → embedding vector
2. Compute cosine similarity against all reference embeddings
3. Take the top-K scores → `similarities` list

```python
import embed, cluster

embedder = embed.get_embedder("clip")
embedder.load(device)

query_embedding = embedder.encode_pil_image(Image.open(query_path))
top_indices, top_values = cluster.perform_search(10, query_embedding, reference_embeddings, reference_paths)
similarities = top_values.cpu().numpy().tolist()  # descending cosine scores
```

**In this notebook**, we use hardcoded `example_similarities` instead — no GPU, no model download, no embeddings file required. The feature extraction and classification logic is identical; only the input data is artificial. Results won't reflect real-world accuracy.

In [ ]:
extractor = FeatureExtractor()

# Distribution features only (fast, no images needed)
# (this would normally be retrieved from actual CLIP model)
example_similarities = [0.82, 0.51, 0.43, 0.41, 0.38, 0.35, 0.33, 0.30, 0.28, 0.25]

dist_features = extractor.extract_distribution_features(example_similarities)

print("Distribution features for example similarities:")
for name, value in dist_features.items():
    print(f"  {name}: {value:.4f}")

In [ ]:
# Full feature extraction (distribution + ORB + hashes) requires image paths
if QUERY_IMAGE.exists() and REFERENCE_IMAGE.exists():
    all_features = extractor.extract_all_features(
        example_similarities,
        glashelder_path=str(QUERY_IMAGE),
        wikiart_path=str(REFERENCE_IMAGE),
    )

    print("All 23 features:")
    for name, value in all_features.items():
        print(f"  {name}: {value:.4f}")
else:
    print(
        f"Place example images at:\n  {QUERY_IMAGE}\n  {REFERENCE_IMAGE}\nto extract ORB and hash features."
    )

## 1.3 Visualize Example Images

In [ ]:
if QUERY_IMAGE.exists() and REFERENCE_IMAGE.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(Image.open(QUERY_IMAGE))
    axes[0].set_title("Query (glashelder)")
    axes[0].axis("off")

    axes[1].imshow(Image.open(REFERENCE_IMAGE))
    axes[1].set_title("Reference (wikiart)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"Place example images at:\n  {QUERY_IMAGE}\n  {REFERENCE_IMAGE}")

---

## 1.4 Classification

Load a trained model and classify an image pair.

> **Note:** this uses `example_similarities` — artificial data. The confidence score below is not meaningful; it just demonstrates the API. Real results require actual embedding similarities from an image pair (see §1.2).

To create a labeled dataset and train a model, see [dataset.ipynb](dataset.ipynb).

In [ ]:
if TRAINED_MODEL_PATH.exists():
    classifier = MLClassifier.load(str(TRAINED_MODEL_PATH), threshold=0.81)
    print(f"Loaded model (threshold={classifier.threshold:.2f})")

    result = classifier.classify(
        similarities=example_similarities,
        glashelder_path=str(QUERY_IMAGE) if QUERY_IMAGE.exists() else None,
        wikiart_path=str(REFERENCE_IMAGE) if REFERENCE_IMAGE.exists() else None,
    )

    print(f"\nIs match: {result.is_match}")
    print(f"Confidence: {result.confidence:.4f}")
    print(f"Threshold: {result.threshold:.4f}")
else:
    print(f"No trained model at {TRAINED_MODEL_PATH}")
    print("See dataset.ipynb to create a labeled dataset and train a model.")

## 1.5 Visualize Feature Importance

If a model is loaded, visualize which features matter most.

In [ ]:
if TRAINED_MODEL_PATH.exists():
    clf = MLClassifier.load(str(TRAINED_MODEL_PATH))

    # Get feature importance from the base RF inside CalibratedClassifierCV
    base_rf = clf._model.calibrated_classifiers_[0].estimator  # ty:ignore[unresolved-attribute]
    importances = base_rf.feature_importances_
    feature_importance = dict(zip(MLClassifier.FEATURE_ORDER, importances, strict=True))

    sorted_features = sorted(feature_importance.items(), key=lambda x: x[1])
    names, importances = zip(*sorted_features, strict=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(names, importances)
    ax.set_xlabel("Importance")
    ax.set_title("Random Forest Feature Importance")
    plt.tight_layout()
    plt.show()
else:
    print(f"No trained model at {TRAINED_MODEL_PATH}")

---

## CLI Commands

```bash
# Classify a single image pair
uv run glashelder artmatcher match <image1> <image2>
```

For dataset creation, labeling, and training — see [dataset.ipynb](dataset.ipynb).